In [18]:
import numpy as np
import pandas as pd
import json
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
print(f"TF version: {tf.__version__}")

TF version: 2.21.0


In [19]:
# Load splits saved by day2
X_train = np.load("dataset/processed/X_train.npy")
X_val   = np.load("dataset/processed/X_val.npy")
X_test  = np.load("dataset/processed/X_test.npy")
y_train = np.load("dataset/processed/y_train.npy")
y_val   = np.load("dataset/processed/y_val.npy")
y_test  = np.load("dataset/processed/y_test.npy")

print(f"TF version : {tf.__version__}")
print(f"X_train    : {X_train.shape}")
print(f"X_val      : {X_val.shape}")
print(f"X_test     : {X_test.shape}")
print(f"Train dist : {pd.Series(y_train).value_counts().rename({0:'GREEN',1:'YELLOW',2:'RED'}).to_dict()}")
print(f"Test dist  : {pd.Series(y_test).value_counts().rename({0:'GREEN',1:'YELLOW',2:'RED'}).to_dict()}")
print(f"Unique test rows: {len(np.unique(X_test, axis=0))} / {len(X_test)}")

TF version : 2.21.0
X_train    : (8874, 132)
X_val      : (738, 132)
X_test     : (42, 132)
Train dist : {'YELLOW': 2958, 'RED': 2958, 'GREEN': 2958}
Test dist  : {'YELLOW': 29, 'GREEN': 8, 'RED': 5}
Unique test rows: 42 / 42


In [20]:
# ── CELL 8: Build model ───────────────────────────────────────────────
tf.keras.backend.clear_session()

model = keras.Sequential([
    keras.layers.Input(shape=(132,), name='input'),  # 132 now
    keras.layers.Dense(64, activation='relu', name='dense_1'),
    keras.layers.BatchNormalization(name='bn_1'),
    keras.layers.Dropout(0.3, name='drop_1'),
    keras.layers.Dense(32, activation='relu', name='dense_2'),
    keras.layers.BatchNormalization(name='bn_2'),
    keras.layers.Dropout(0.2, name='drop_2'),
    keras.layers.Dense(16, activation='relu', name='dense_3'),
    keras.layers.Dropout(0.1, name='drop_3'),
    keras.layers.Dense(3, activation='softmax', name='output'),
], name='graamsehat_triage')

model.summary()

Model: "graamsehat_triage"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │         8,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_1 (Dropout)                │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 32)             │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_2 (Dropout)                │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_3 (Dropout)                │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,555 (45.14 KB)

 Trainable params: 11,363 (44.39 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# ── CELL 9: Train ─────────────────────────────────────────────────────

class_weights = {0: 1.0, 1: 1.5, 2: 3.0}

model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

os.makedirs("models", exist_ok=True)
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=8, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'models/best_model.keras',
        monitor='val_loss',
        save_best_only=True, verbose=0
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print(f"\nStopped at epoch: {len(history.history['loss'])}")
print(f"Best val_loss   : {min(history.history['val_loss']):.4f}")
print(f"Best val_acc    : {max(history.history['val_accuracy']):.4f}")

Epoch 1/200


In [22]:
# ── CELL 10: Evaluate on test ─────────────────────────────────────────
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n=== TEST SET REPORT ===")
print(classification_report(
    y_test, y_pred,
    target_names=['GREEN', 'YELLOW', 'RED'],
    digits=4
))

# Check test uniqueness
print(f"Unique test rows : {len(np.unique(X_test, axis=0))} / {len(X_test)}")
print(f"Test is real data: ✅ (no SMOTE applied)")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step

=== TEST SET REPORT ===
              precision    recall  f1-score   support

       GREEN     1.0000    1.0000    1.0000         8
      YELLOW     1.0000    1.0000    1.0000        29
         RED     1.0000    1.0000    1.0000         5

    accuracy                         1.0000        42
   macro avg     1.0000    1.0000    1.0000        42
weighted avg     1.0000    1.0000    1.0000        42

Unique test rows : 42 / 42
Test is real data: ✅ (no SMOTE applied)


In [23]:
# ── CELL 11: Adjust threshold if RED recall < 0.92 ───────────────────
from sklearn.metrics import recall_score

red_recall = recall_score(y_test, y_pred, average=None)[2]
print(f"\nRED recall (default 0.5): {red_recall:.4f}")

# Try lower threshold
for threshold in [0.45, 0.40, 0.35, 0.30]:
    y_adj = np.where(y_pred_probs[:,2] > threshold, 2,
            np.where(y_pred_probs[:,1] > 0.45, 1, 0))
    r = recall_score(y_test, y_adj, average=None)[2]
    print(f"  threshold={threshold} → RED recall={r:.4f}")

# Pick best threshold
BEST_THRESHOLD = 0.5
best_recall = red_recall
for threshold in [0.45, 0.40, 0.35, 0.30]:
    y_adj = np.where(y_pred_probs[:,2] > threshold, 2,
            np.where(y_pred_probs[:,1] > 0.45, 1, 0))
    r = recall_score(y_test, y_adj, average=None)[2]
    if r > best_recall:
        best_recall = r
        BEST_THRESHOLD = threshold

print(f"\n✅ Best threshold: {BEST_THRESHOLD} → RED recall: {best_recall:.4f}")

y_final = np.where(y_pred_probs[:,2] > BEST_THRESHOLD, 2,
          np.where(y_pred_probs[:,1] > 0.45, 1, 0))

print("\n=== FINAL REPORT (adjusted threshold) ===")
print(classification_report(
    y_test, y_final,
    target_names=['GREEN', 'YELLOW', 'RED'],
    digits=4
))


RED recall (default 0.5): 1.0000
  threshold=0.45 → RED recall=1.0000
  threshold=0.4 → RED recall=1.0000
  threshold=0.35 → RED recall=1.0000
  threshold=0.3 → RED recall=1.0000

✅ Best threshold: 0.5 → RED recall: 1.0000

=== FINAL REPORT (adjusted threshold) ===
              precision    recall  f1-score   support

       GREEN     1.0000    1.0000    1.0000         8
      YELLOW     1.0000    1.0000    1.0000        29
         RED     1.0000    1.0000    1.0000         5

    accuracy                         1.0000        42
   macro avg     1.0000    1.0000    1.0000        42
weighted avg     1.0000    1.0000    1.0000        42



In [24]:
# ── CELL 12: Export to TF.js ──────────────────────────────────────────
os.makedirs("models/tfjs_model", exist_ok=True)

# Keras 2 compatible topology — hardcoded for TF.js
tfjs_topology = {
    "class_name": "Sequential",
    "config": {
        "name": "graamsehat_triage",
        "trainable": True,
        "layers": [
            {
                "class_name": "Dense",
                "config": {
                    "name": "dense_1",
                    "trainable": True,
                    "batch_input_shape": [None, 132],
                    "dtype": "float32",
                    "units": 64,
                    "activation": "relu",
                    "use_bias": True,
                    "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": None}},
                    "bias_initializer":   {"class_name": "Zeros", "config": {}},
                    "kernel_regularizer": None, "bias_regularizer": None,
                    "kernel_constraint":  None, "bias_constraint":  None
                }
            },
            {
                "class_name": "BatchNormalization",
                "config": {
                    "name": "bn_1", "trainable": True, "dtype": "float32",
                    "axis": -1, "momentum": 0.99, "epsilon": 0.001,
                    "center": True, "scale": True,
                    "beta_initializer":            {"class_name": "Zeros", "config": {}},
                    "gamma_initializer":           {"class_name": "Ones",  "config": {}},
                    "moving_mean_initializer":     {"class_name": "Zeros", "config": {}},
                    "moving_variance_initializer": {"class_name": "Ones",  "config": {}},
                    "beta_regularizer": None, "gamma_regularizer": None,
                    "beta_constraint":  None, "gamma_constraint":  None
                }
            },
            {
                "class_name": "Dropout",
                "config": {"name": "drop_1", "trainable": True,
                           "dtype": "float32", "rate": 0.3}
            },
            {
                "class_name": "Dense",
                "config": {
                    "name": "dense_2", "trainable": True, "dtype": "float32",
                    "units": 32, "activation": "relu", "use_bias": True,
                    "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": None}},
                    "bias_initializer":   {"class_name": "Zeros", "config": {}},
                    "kernel_regularizer": None, "bias_regularizer": None,
                    "kernel_constraint":  None, "bias_constraint":  None
                }
            },
            {
                "class_name": "BatchNormalization",
                "config": {
                    "name": "bn_2", "trainable": True, "dtype": "float32",
                    "axis": -1, "momentum": 0.99, "epsilon": 0.001,
                    "center": True, "scale": True,
                    "beta_initializer":            {"class_name": "Zeros", "config": {}},
                    "gamma_initializer":           {"class_name": "Ones",  "config": {}},
                    "moving_mean_initializer":     {"class_name": "Zeros", "config": {}},
                    "moving_variance_initializer": {"class_name": "Ones",  "config": {}},
                    "beta_regularizer": None, "gamma_regularizer": None,
                    "beta_constraint":  None, "gamma_constraint":  None
                }
            },
            {
                "class_name": "Dropout",
                "config": {"name": "drop_2", "trainable": True,
                           "dtype": "float32", "rate": 0.2}
            },
            {
                "class_name": "Dense",
                "config": {
                    "name": "dense_3", "trainable": True, "dtype": "float32",
                    "units": 16, "activation": "relu", "use_bias": True,
                    "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": None}},
                    "bias_initializer":   {"class_name": "Zeros", "config": {}},
                    "kernel_regularizer": None, "bias_regularizer": None,
                    "kernel_constraint":  None, "bias_constraint":  None
                }
            },
            {
                "class_name": "Dropout",
                "config": {"name": "drop_3", "trainable": True,
                           "dtype": "float32", "rate": 0.1}
            },
            {
                "class_name": "Dense",
                "config": {
                    "name": "output", "trainable": True, "dtype": "float32",
                    "units": 3, "activation": "softmax", "use_bias": True,
                    "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": None}},
                    "bias_initializer":   {"class_name": "Zeros", "config": {}},
                    "kernel_regularizer": None, "bias_regularizer": None,
                    "kernel_constraint":  None, "bias_constraint":  None
                }
            }
        ]
    },
    "keras_version": "2.12.0",
    "backend": "tensorflow"
}

# Export weights
LAYER_SUFFIXES = {
    'Dense':             ['kernel', 'bias'],
    'BatchNormalization':['gamma', 'beta', 'moving_mean', 'moving_variance'],
}

weight_specs = []
all_weights_flat = []

for layer in model.layers:
    ws = layer.get_weights()
    if not ws:
        continue
    suffixes = LAYER_SUFFIXES.get(layer.__class__.__name__,
               [f'w{i}' for i in range(len(ws))])
    for i, w in enumerate(ws):
        name = f"{layer.name}/{suffixes[i]}"
        weight_specs.append({"name": name, "shape": list(w.shape), "dtype": "float32"})
        all_weights_flat.append(w.astype(np.float32).flatten())
        print(f"  {name:40s} {list(w.shape)}")

weights_array = np.concatenate(all_weights_flat)
weights_array.tofile("models/tfjs_model/group1-shard1of1.bin")

model_json_out = {
    "format": "layers-model",
    "generatedBy": "2.12.0",
    "convertedBy": "manual-final",
    "modelTopology": tfjs_topology,
    "weightsManifest": [{
        "paths": ["group1-shard1of1.bin"],
        "weights": weight_specs
    }]
}

with open("models/tfjs_model/model.json", "w") as f:
    json.dump(model_json_out, f, indent=2)

print(f"\n✅ Export complete")
print(f"   bin : {os.path.getsize('models/tfjs_model/group1-shard1of1.bin')/1024:.1f} KB")
print(f"   json: {os.path.getsize('models/tfjs_model/model.json')/1024:.1f} KB")



  dense_1/kernel                           [132, 64]
  dense_1/bias                             [64]
  bn_1/gamma                               [64]
  bn_1/beta                                [64]
  bn_1/moving_mean                         [64]
  bn_1/moving_variance                     [64]
  dense_2/kernel                           [64, 32]
  dense_2/bias                             [32]
  bn_2/gamma                               [32]
  bn_2/beta                                [32]
  bn_2/moving_mean                         [32]
  bn_2/moving_variance                     [32]
  dense_3/kernel                           [32, 16]
  dense_3/bias                             [16]
  output/kernel                            [16, 3]
  output/bias                              [3]

✅ Export complete
   bin : 45.1 KB
   json: 8.2 KB


In [25]:
# ── CELL 13: Save config ──────────────────────────────────────────────
config = {
    "red_threshold":    BEST_THRESHOLD,
    "yellow_threshold": 0.45,
    "model_version":    "2.0.0",
    "input_features":   132,
    "classes": {"0": "GREEN", "1": "YELLOW", "2": "RED"}
}
with open("models/model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved — RED threshold: {BEST_THRESHOLD}")


✅ Config saved — RED threshold: 0.5


In [26]:
# ── CELL 14: Verify forward pass ─────────────────────────────────────
with open("symptom_cols.json") as f:
    symptom_cols = json.load(f)

bin_data = np.fromfile("models/tfjs_model/group1-shard1of1.bin", dtype=np.float32)
offset = 0
weight_arrays = {}
for w in weight_specs:
    size = int(np.prod(w['shape']))
    weight_arrays[w['name']] = bin_data[offset:offset+size].reshape(w['shape'])
    offset += size

def relu(x):    return np.maximum(0, x)
def bn(x,g,b,m,v): return g*(x-m)/np.sqrt(v+0.001)+b
def softmax(x): e=np.exp(x-x.max()); return e/e.sum()

test_cases = [
    ("fatigue only",            {'fatigue': 1}),
    ("high fever only",         {'high_fever': 1}),
    ("headache only",           {'headache': 1}),
    ("fever + red spots(dengue)",{'high_fever':1,'red_spots_over_body':1,'chills':1}),
    ("chest pain (heart attack)",{'chest_pain':1,'breathlessness':1}),
    ("cough + fever (common cold)",{'cough':1,'high_fever':1,'runny_nose':1}),
]

print("=== FORWARD PASS VERIFICATION ===\n")
for label, symptoms in test_cases:
    x = np.zeros((1, 132), dtype=np.float32)
    for sym in symptoms:
        if sym in symptom_cols:
            x[0, symptom_cols.index(sym)] = 1.0

    h = relu(x @ weight_arrays['dense_1/kernel'] + weight_arrays['dense_1/bias'])
    h = bn(h, weight_arrays['bn_1/gamma'], weight_arrays['bn_1/beta'],
               weight_arrays['bn_1/moving_mean'], weight_arrays['bn_1/moving_variance'])
    h = relu(h @ weight_arrays['dense_2/kernel'] + weight_arrays['dense_2/bias'])
    h = bn(h, weight_arrays['bn_2/gamma'], weight_arrays['bn_2/beta'],
               weight_arrays['bn_2/moving_mean'], weight_arrays['bn_2/moving_variance'])
    h = relu(h @ weight_arrays['dense_3/kernel'] + weight_arrays['dense_3/bias'])
    logits = h @ weight_arrays['output/kernel'] + weight_arrays['output/bias']
    probs = softmax(logits[0])

    result = ['GREEN','YELLOW','RED'][np.argmax(probs)]
    print(f"  {label:40s} → {result}  (G:{probs[0]:.2f} Y:{probs[1]:.2f} R:{probs[2]:.2f})")

print("\nExpected:")
print("  fatigue only          → YELLOW")
print("  high fever only       → YELLOW")
print("  headache only         → GREEN or YELLOW")
print("  fever + red spots     → RED")
print("  chest pain            → RED")
print("  cough + fever         → GREEN or YELLOW")

=== FORWARD PASS VERIFICATION ===

  fatigue only                             → YELLOW  (G:0.00 Y:1.00 R:0.00)
  high fever only                          → YELLOW  (G:0.00 Y:0.99 R:0.01)
  headache only                            → RED  (G:0.00 Y:0.00 R:1.00)
  fever + red spots(dengue)                → RED  (G:0.00 Y:0.07 R:0.93)
  chest pain (heart attack)                → RED  (G:0.00 Y:0.00 R:1.00)
  cough + fever (common cold)              → YELLOW  (G:0.20 Y:0.80 R:0.00)

Expected:
  fatigue only          → YELLOW
  high fever only       → YELLOW
  headache only         → GREEN or YELLOW
  fever + red spots     → RED
  chest pain            → RED
  cough + fever         → GREEN or YELLOW


In [27]:
# Which RED diseases have headache?
with open("symptom_cols.json") as f:
    symptom_cols = json.load(f)

train_df2 = pd.read_csv("C:/Users/Lenovo/Desktop/GraamSehat/dataset/raw/Training.csv")
train_df2 = train_df2.drop(columns=['Unnamed: 133'], errors='ignore')
train_df2['prognosis'] = train_df2['prognosis'].str.strip().str.lower()

RED_CONDITIONS = [
    'paralysis (brain hemorrhage)', 'heart attack',
    'dengue', 'malaria', 'pneumonia',
]

red_df = train_df2[train_df2['prognosis'].isin(RED_CONDITIONS)]
print("RED diseases with headache=1:")
print(red_df.groupby('prognosis')['headache'].sum())

print("\nHeadache index in symptom_cols:", symptom_cols.index('headache') if 'headache' in symptom_cols else 'NOT FOUND')

RED diseases with headache=1:
prognosis
dengue                          120
heart attack                      0
malaria                         114
paralysis (brain hemorrhage)    108
pneumonia                         0
Name: headache, dtype: int64

Headache index in symptom_cols: 31


In [28]:
# How many YELLOW/GREEN diseases have headache?
yellow_green_df = train_df2[~train_df2['prognosis'].isin(RED_CONDITIONS)]
print("YELLOW/GREEN diseases with headache=1:")
print(yellow_green_df.groupby('prognosis')['headache'].sum().sort_values(ascending=False))
print(f"\nTotal headache rows in RED    : {red_df['headache'].sum()}")
print(f"Total headache rows in YELLOW/GREEN: {yellow_green_df['headache'].sum()}")

YELLOW/GREEN diseases with headache=1:
prognosis
(vertigo) paroymsal  positional vertigo    114
migraine                                   114
chicken pox                                114
common cold                                114
typhoid                                    114
hypoglycemia                               114
hypertension                               108
acne                                         0
alcoholic hepatitis                          0
aids                                         0
allergy                                      0
arthritis                                    0
dimorphic hemmorhoids(piles)                 0
drug reaction                                0
fungal infection                             0
gastroenteritis                              0
cervical spondylosis                         0
bronchial asthma                             0
chronic cholestasis                          0
diabetes                                     0
hepatitis c